In [2]:

import matplotlib
import jax
from flax import nnx
import jax.numpy as jnp
import math
import optax
import gc

ModuleNotFoundError: No module named 'jax'

In [65]:
class SelfAttention(nnx.Module):
  def __init__(self, in_features, qKV_features, out_features, rngs: nnx.Rngs):
    self.w_q = nnx.Linear(qKV_features, use_bias=False, rngs=rngs)
    self.w_k = nnx.Linear(qKV_features, use_bias=False, rngs=rngs)
    self.w_v = nnx.Linear(qKV_features, use_bias=False, rngs=rngs)

  def __call__(self, x):
    queries = self.w_q(x)
    keys = self.w_k(x)
    values = self.w_v(x)

    # q.(k.T)
    alignment_scores = queries @ keys.T

    # scaled
    alignment_scores = alignment_scores / math.sqrt(queries.shape[-1])

    # masked, lower triangle set to -inf before softmax
    mask = jnp.tril(jnp.ones(alignment_scores.shape))
    alignment_scores = jnp.where(mask == 0, alignment_scores, -jnp.inf)

    # softmax
    alignment_scores = jax.nn.softmax(alignment_scores)

    # aligment_scores . values
    attention_values = alignment_scores @ values

    return attention_values

In [66]:
class MultiHeadAttention(nnx.Module):
  def __init__(self, n_heads, in_features, rngs: nnx.Rngs):
    self.n_heads = n_heads
    self.head_dim = in_features // n_heads

    # input: Batch, context_len, embed_dim
    # MultiHead Attention
    #   so each head takes slice of the embeddings
    #   for each token in context as input
    #   therefore, for each head dimensions will be
    #   input: Batch, context_len, embed_dim// n_heads


    self.q_proj = nnx.Linear(in_features, in_features, use_bias=False, rngs=rngs)
    self.k_proj = nnx.Linear(in_features, in_features, use_bias=False, rngs=rngs)
    self.v_proj = nnx.Linear(in_features, in_features, use_bias=False, rngs=rngs)

    self.out_proj = nnx.Linear(in_features, in_features, use_bias=False, rngs=rngs)

  def __call__(self, input_ids, attention_mask):
    # print("MultiHeadAttention")
    # print(f"input_ids: {input_ids.shape}")
    # print(f"attention_mask: {attention_mask.shape}")
    x = input_ids * attention_mask

    # print(f"x: {x.shape}")

    # x: Batch, Context len, Embed dim
    B, L, E = x.shape
    assert E == self.n_heads * self.head_dim, f"embed_dim: ({E}), must be divisible by n_heads: ({self.n_heads})"

    queries = self.q_proj(x)
    keys = self.k_proj(x)
    values = self.v_proj(x)

    # print(f"queries: {queries.shape}")
    # print(f"keys: {keys.shape}")
    # print(f"values: {values.shape}")
    queries = queries.reshape(B, L, self.n_heads, self.head_dim)
    keys = keys.reshape(B, L, self.n_heads, self.head_dim)
    values = values.reshape(B, L, self.n_heads, self.head_dim)

    # could use swapaxes, the following reads better to me
    queries = queries.transpose(0, 2, 1, 3)
    keys = keys.transpose(0, 2, 1, 3)
    values = values.transpose(0, 2, 1, 3)

    # print(f"queries: {queries.shape}")
    # print(f"keys: {keys.shape}")
    # print(f"values: {values.shape}")

    alignment_scores = queries @ keys.transpose(0, 1, 3, 2)
    alignment_scores = alignment_scores / math.sqrt(queries.shape[-1])

    mask = jnp.tril(jnp.ones(alignment_scores.shape))
    alignment_scores = jnp.where(mask == 0, alignment_scores, -jnp.inf)

    alignment_scores = jax.nn.softmax(alignment_scores)

    attention_values = alignment_scores @ values
    attention_values = attention_values.transpose(0, 2, 1, 3)

    # print(f"attention_values: {attention_values.shape}")
    attention_values = attention_values.reshape(B, L, E)
    # print(f"attention_values: {attention_values.shape}")

    attention_values = self.out_proj(attention_values)

    # print(f"attention_values: {attention_values.shape}")
    return attention_values

In [67]:
class MLP(nnx.Module):
  def __init__(self, in_features, hidden_features, out_features, rngs: nnx.Rngs):
    self.w1 = nnx.Linear(in_features, hidden_features, rngs=rngs)
    self.w2 = nnx.Linear(hidden_features, out_features, rngs=rngs)

  def __call__(self, x):
    x = jax.nn.gelu(self.w1(x))
    x = self.w2(x)
    return x

In [68]:
class Transformer(nnx.Module):
  def __init__(self, embed_dim, n_heads, mlp_hidden_dim, rngs: nnx.Rngs):
    self.attention = MultiHeadAttention(n_heads, embed_dim, rngs)
    self.mlp = MLP(embed_dim, mlp_hidden_dim, embed_dim, rngs)

    self.ln1 = nnx.LayerNorm(embed_dim, rngs=rngs)
    self.ln2 = nnx.LayerNorm(embed_dim, rngs=rngs)

  def __call__(self, input_ids, attention_mask):
    x = input_ids
    norm_x = self.ln1(x)
    attention = self.attention(norm_x, attention_mask)

    x = x + attention
    norm_x = self.ln2(x)

    mlp_output = self.mlp(norm_x)

    return mlp_output + x


In [69]:
class Model(nnx.Module):
  def __init__(self, embed_dim, seq_len, vocab_size, n_heads, mlp_hidden_dim, n_layers, rngs: nnx.Rngs):
    self.embed_dim = embed_dim
    self.n_heads = n_heads
    self.mlp_hidden_dim = mlp_hidden_dim
    self.n_layers = n_layers

    self.seq_len = seq_len
    self.vocab_size = vocab_size

    self.token_embedding = nnx.Embed(vocab_size, embed_dim, rngs=rngs)
    self.position_embedding = nnx.Embed(seq_len, embed_dim, rngs=rngs)

    self.layers = [Transformer(embed_dim, n_heads, mlp_hidden_dim, rngs=rngs) for _ in range(n_layers)]

    self.ln = nnx.LayerNorm(embed_dim, rngs=rngs)
    self.head = nnx.Linear(embed_dim, vocab_size, use_bias=False, rngs=rngs)

  def __call__(self, input_ids, attention_mask):
    # input_ids: (Batch, SeqLen)
    # attention_mask: (Batch, SeqLen)
    # print(f"model input_ids: {input_ids.shape}")
    # print(f"model attention_mask: {attention_mask.shape}")
    attention_mask = jnp.expand_dims(attention_mask, axis=-1)
    t_embeds = self.token_embedding(input_ids.astype(jnp.int32)) # (Batch, SeqLen, EmbedDim)
    p_embeds = self.position_embedding(jnp.arange(input_ids.shape[1])) # (SeqLen, EmbedDim)

    x = t_embeds + p_embeds

    for layer in self.layers:
      x = layer(x, attention_mask) # Pass embedded sequence 'x' and mask to transformer layer

    norm_x = self.ln(x)

    logits = self.head(norm_x)

    return logits

In [70]:
def loss(model, inputs, targets):
  # inputs are Batch, SeqLen but dicts with input_ids and attention_mask
  input_ids = []
  attention_mask = []
  for batch in inputs:
    input_ids.append(batch['input_ids'])
    attention_mask.append(batch['attention_mask'])

  input_ids = jnp.array(input_ids)
  attention_mask = jnp.array(attention_mask)

  # print("loss")
  # print(f"input_ids: {input_ids.shape}")
  # print(f"attention_mask: {attention_mask.shape}")
  logits = model(input_ids, attention_mask)

  target_ids = []
  target_attention_masks = []

  # TODO: find a better way to do this;
  # jnp maybe?
  for batch in targets:
    target_ids.append(batch['input_ids'])
    target_attention_masks.append(batch['attention_mask'])

  target_ids = jnp.array(target_ids)
  target_attention_masks = jnp.array(target_attention_masks)

  targets = target_ids * target_attention_masks
  targets = targets.astype(jnp.int32)
  # print(f"logits: {logits.shape}")
  # print(f"targets: {targets.shape}")

  # print(f"logits dtypef: {logits.dtype}")
  # print(f"targets dtype: {targets.dtype}")
  loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets)

  del logits, targets, input_ids, attention_mask, target_ids, target_attention_masks
  return loss.mean()

def train_step(optimizer, model, inputs, targets):
  # print time taken for each training step
  start_time = jax.process_index() * jax.process_count()

  loss_value, grads = jax.value_and_grad(loss)(model, inputs, targets)
  model = optimizer.update(model, grads)

  del loss_value, grads

  end_time = jax.process_index() * jax.process_count()
  print(f"Time taken for traininkg step: {end_time - start_time} seconds")

  gc.collect()

  return loss_value, model

def choose_from_logits(logits):
  # print(f"logits during selection: {logits.shape}")
  # logits: (Batch, Seq_len, Vocab_size)
  # from the last tokens predictions
  logits = logits[:, -1, :]
  # select one with max probab
  return jnp.argmax(logits, axis=-1, keepdims=True)

def generate(model, inputs, tokenizer, n_tokens):
  outputs = []
  input_ids = inputs[0]['input_ids']
  attention_mask = jnp.ones_like(input_ids)

  input_ids = jnp.expand_dims(input_ids, axis=0)
  attention_mask = jnp.expand_dims(attention_mask, axis=0)

  # print("generate")
  # print(f"input_ids: {input_ids.shape}")
  # print(f"attention_mask: {attention_mask.shape}")

  for _ in range(n_tokens):
    logits = model(input_ids, attention_mask)
    next_token = choose_from_logits(logits)

    # print(f"next_token: {next_token.shape}")
    # print(f"next_token: {next_token}")

    # add to inputs
    inputs = jnp.concatenate([input_ids, next_token], axis=1)

    outputs.append(next_token)

  outputs = jnp.array(outputs).reshape(-1)

  del input_ids
  del attention_mask
  del logits
  del next_token
  # print(f"outputs: {outputs.shape}")
  return tokenizer.decode(outputs)

def generate_targets_from_input(inputs):
  # takes in batches of input_ids and attention_maskes
  # returns inputs, targets for autoregressive task

  # input_batches: [{input_ids, attention_masks}]
  # target_batches: [{input_ids, attention_masks}]

  input_batches = []
  target_batches = []
  for batch in inputs:
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']

    targets_ids = jnp.array(input_ids[1:], dtype = jnp.float32)
    inputs_ids = jnp.array(input_ids[:-1], dtype = jnp.float32)

    targets_attention_mask = jnp.array(attention_mask[1:], dtype = jnp.float32)
    inputs_attention_mask = jnp.array(attention_mask[:-1], dtype = jnp.float32)

    input_batches.append({'input_ids': inputs_ids, 'attention_mask': inputs_attention_mask})
    target_batches.append({'input_ids': targets_ids, 'attention_mask': targets_attention_mask})

  return input_batches, target_batches

def train(model,  raw_inputs_batches, raw_validation_batches, n_steps):
  # generate targets from inputs, shift both attention_mask and input_ids
  inputs, targets = generate_targets_from_input(raw_inputs_batches)
  val_inputs, val_targets = generate_targets_from_input(raw_validation_batches)

  gc.collect()
  del raw_inputs_batches
  del raw_validation_batches

  optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=3e-4), wrt=nnx.Param)

  for i in range(n_steps):
    # Validation step (every 10 steps)
    # if i % 10 == 0:
    #   avg_val_loss = loss(model, val_inputs, val_targets)
    #   print(f"Validation loss: {avg_val_loss}")

    #   # Generate also needs the input_ids array, not the dictionary.
    #   # Use the 'input_ids' from the first validation input batch for generation
    #   print(generate(model, val_inputs, tokenizer, 10))

    # Training step for each epoch
    current_step_losses = []

    loss_value, model = train_step(optimizer, model, inputs, targets)
    current_step_losses.append(loss_value)

    avg_train_loss = jnp.mean(jnp.array(current_step_losses))
    print(f"Step: {i}, Loss: {avg_train_loss}")

  return model

In [71]:
n_heads = 8
embed_dim = n_heads * 18
seq_len = 10
vocab_size = 1000
mlp_hidden_dim = 512
n_layers = 6
model = Model(embed_dim=embed_dim, seq_len=seq_len, vocab_size=vocab_size, n_heads=n_heads, mlp_hidden_dim=mlp_hidden_dim, n_layers=n_layers, rngs=nnx.Rngs(42))

In [72]:
nnx.display(model)

In [73]:
!pip install transformers

In [74]:
from transformers import AutoTokenizer as AT

In [75]:
# fetch tiny shakespeare dataset and store in variable for processing
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()
  f.close()

print(f"Dataset len: {len(text)}")
print(text[:100])

--2026-08-03 01:17:31--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-08-03 01:17:31 (36.3 MB/s) - ‘input.txt’ saved [1115394/1115394]

Dataset len: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [76]:
from huggingface_hub import login
login()

In [77]:
tokenizer = AT.from_pretrained('google/gemma-2-2b', vocab_size=vocab_size)

In [78]:
inputs = tokenizer(text)

del text

len(inputs)

2

In [79]:
len(inputs.input_ids)

314936

In [80]:
len(set(inputs.input_ids))

14209

In [81]:
def generate_batches(inputs, batch_size):
  # inputs is dict with input_ids and attention_mask
  # both arrays should be distributed in the batch
  # add one more dimension of Batch
  num_batches = len(inputs.input_ids) // batch_size

  batches = []
  for i in range(num_batches):
    batch_input_ids = jnp.array(inputs.input_ids[i * batch_size : (i + 1) * batch_size])
    batch_attention_mask = jnp.array(inputs.attention_mask[i * batch_size : (i + 1) * batch_size])

    batches.append({"input_ids": batch_input_ids, "attention_mask": batch_attention_mask})

  return batches

In [82]:
batch_size = 10
batches = generate_batches(inputs, batch_size)
del inputs
print(f"Total batches: {len(batches)}")

Total batches: 31493


In [83]:
# train test split the batches
train_batches = batches[:int(len(batches) * 0.9)]
val_batches = batches[int(len(batches) * 0.9):]

# force variable to be garbage collected
del batches

# convert to suitable input for nnx.Module
len(train_batches[0]['input_ids'])

10

In [84]:
gc.collect()
train(model, train_batches, val_batches, 10)

ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 146930112 bytes.